In [ ]:
# ar/statistics-101/hard/18-multiple-regression-and-confounding
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


## ماذا يعني

يركّب **الانحدار المتعدد** (multiple regression) نموذجًا بأكثر من منبئ واحد:

$$
\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots
$$

يُقرأ كل معامل مع جملة ملازمة: *مع تثبيت المنبئات الأخرى*، ترتبط زيادة وحدة واحدة في $x_1$ بتغير $b_1$ وحدة في $y$. تلك الجملة هي بيت القصيد. فإدراج $x_2$ يزيل من $b_1$ الجزء الذي كان يمكن نسبته إلى $x_2$ أيضًا.

هذا هو الشكل العددي للتحليل المجمّع في درس مفارقة سيمبسون. الدمج دون تعديل ينتج الرقم المربك؛ والتعديل ينتج رقم ما داخل المجموعة.

## لماذا يضيف شيئًا

إنه حصان العمل في الإحصاء التطبيقي. لا تكاد توجد علاقة حقيقية نظيفة بما يكفي لقراءتها من انحدار أحادي المتغير، والإبلاغ عن معامل غير معدَّل هو الطريق الذي تنتج به البيانات الرصدية هراءً واثقًا.

يمنحك أيضًا طريقة لتحديد *ما تحكمت فيه*، وهذه جملة أنفع بكثير من "وجدنا ارتباطًا".

## مثال محسوب

بيانات على نمط القبول، برنامجان ومستويا قوة متقدمين:

| البرنامج | متقدمون أقوياء | مُقبلون | متقدمون ضعفاء | مُقبلون |
|---|---|---|---|---|
| أ | 20 | 18 | 80 | 24 |
| ب | 90 | 72 | 10 | 2 |

**الخطوة 1 — مقارنة غير معدَّلة.** البرنامج أ: 42 مقبولًا من 100، إذن 42%. والبرنامج ب: 74 من 100، إذن 74%. يبدو ب أرحب بكثير.

**الخطوة 2 — داخل كل مجموعة قوة.** بين المتقدمين الأقوياء يقبل أ 90% ويقبل ب 80%. وبين الضعفاء يقبل أ 30% ويقبل ب 20%. أ أفضل في كلتا الحالتين.

**الخطوة 3 — الميكانيزم.** تلقى ب متقدمين أقوياء أكثر بكثير (90 مقابل 20)، والمتقدمون الأقوياء يُقبلون بمعدل أعلى في كل مكان. معدل ب المدمج مرتفع بسبب *من تقدم*، لا بسبب كيفية معاملتهم.

**الخطوة 4 — عدّل.** ارجع القبول على البرنامج *و*قوة المتقدم. يقيس معامل البرنامج الآن الفرق بين متقدمين متساويين في القوة: فينقلب إشارةً لصالح أ.


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.DataFrame({
    'programme': ['A']*100 + ['B']*100,
    'strength':  ['strong']*20 + ['weak']*80 + ['strong']*90 + ['weak']*10,
    'admitted':  [1]*18 + [0]*2 + [1]*24 + [0]*56 + [1]*72 + [0]*18 + [1]*2 + [0]*8,
})

# دون ضبط: البرنامج فقط
print(smf.logit('admitted ~ C(programme)', df).fit().params)

# بعد التعديل: البيانات نفسها مع الضبط على قوة المتقدم
print(smf.logit('admitted ~ C(programme) + C(strength)', df).fit().params)


ينتقل معامل البرنامج من موجب قوي لصالح ب إلى سالب. الصفوف نفسها، والحساب نفسه؛ السؤال وحده تغير.

## العامل المربك والوسيط والمصادم

- **العامل المربك** (confounder) يؤثر في المنبئ والمخرَج معًا. والتعديل لمثله هو الخطوة الصحيحة — قوة المتقدم في المثال أعلاه.
- **الوسيط** (mediator) يقع *على المسار* بين المنبئ والمخرَج (علاج ثم شفاء ثم بقاء). والتعديل لمثله يزيل جزءًا من الأثر الذي تقيسه نفسه، فهو خاطئ عادةً.
- **المصادم** (collider) ينتج عن الاثنين معًا (عرضان ينتجهما المرض نفسه). والتعديل لمثله قد *يخلق* ارتباطًا غير موجود.

التمييز بنيوي، ولا يمكن قراءته من البيانات. يتطلب معرفة ما يسبب ماذا.

<details class="challenge">
<summary>🧩 تحدٍّ، فكّر أولًا ثم أظهر</summary>
<div class="challenge__body">

تجد دراسة أن المرضى المدخلين إلى المستشفى الذين مارسوا رياضة أكثر كانت نتائجهم أسوأ. هل تستنتج أن الرياضة ضارة؟

<p class="challenge__answer">💡 <strong>الجواب:</strong> لا. الإدخال إلى المستشفى مصادم هنا: المرض الشديد والرياضة كلاهما قد يؤدي إليه. فالاشتراط على الإدخال يختار مجموعة يصبح فيها المتغيران مرتبطين في اتجاه معاكس للواقع. لقد عدّل التحليل لمتغير خاطئ بفعل اختيار العينة.</p>

</div>
</details>

## متى تستخدمه

- مقارنة مجموعات تختلف بطريقة قابلة للقياس تستطيع التحكم فيها.
- الإبلاغ عن أثر كان القارئ سيدعوه مربكًا لولا ذلك.
- التحقق من بقاء نتيجة غير معدَّلة بعد التعديل؛ فإذا تحرك المعامل كثيرًا، فالرقم غير المعدَّل كان يقيس شيئًا آخر.

## القيود

- **يعدّل فقط لما قسته.** يبقى التربيك غير المقاس، وهو عادةً التهديد المهيمن في العمل الرصدي.
- **التعديل لوسيط أو مصادم يضيف تحيزًا** بدلًا من إزالته.
- **الخطية المتعددة** بين المنبئات تجعل المعاملات غير مستقرة ويصعب تفسيرها منفصلة.
- **شكل الدالة مهم.** تركيب خطوط مستقيمة بينما تنحني العلاقة يبلغ خطأً عن كل معامل.
- **المعامل المعدَّل ليس أثرًا سببيًا** ما لم يبرره التصميم.

## أخطاء شائعة

- **التعديل لكل ما لديك.** المزيد من المتغيرات المشتركة ليس مزيدًا من الصرامة؛ وكل واحد منها يحتاج تبريرًا.
- **قراءة معامل دون جملته "مع تثبيت البقية".**
- **إسقاط متغير لأنه غير دال**، مما قد يحيّز البقية.
- **الإبلاغ عن النموذج المعدَّل فقط**، مما يخفي مقدار تحريك التعديل للإجابة.

## 🧩 تحديات

<details class="challenge">
<summary>🧩 تحدٍّ، فكّر أولًا ثم أظهر</summary>
<div class="challenge__body">

معامل غير معدَّل لبرنامج تدريبي هو +4 نقاط. وبعد التعديل للقدرة السابقة يصبح +1. ما القراءتان الممكنتان، وكيف تميز بينهما؟

<p class="challenge__answer">💡 <strong>الجواب:</strong> إما أن الفائدة الظاهرة للبرنامج كانت انتقاءً في معظمها (سجل أشخاص أكثر قدرة، وقدرتهم هي التي حملت النقاط الأربع)، أو أن القدرة السابقة وسيط يحسّنه البرنامج. ميّز بسؤال: هل قيست القدرة السابقة قبل بدء البرنامج؟ فإن كانت سابقة، فهي عامل مربك و+1 هو التقدير الصادق؛ وإن قيست بعده، فقد تقع على المسار السببي والتعديل يزيل أثرًا حقيقيًا.</p>

</div>
</details>

## 🤔 أسئلة سقراطية

- لماذا يزيل المعامل المعدَّل أثر العامل المربك لا الوسيط؟
- إذا غيّر التعديل معاملًا من +4 إلى +1، فأي الرقمين تبلغ؟
- هل يمكن أن يكون منبئان مترابطين لدرجة تصبح معهما المعاملان غير قابلين للتفسير؟ ماذا تنظر إليه؟
- لماذا قد يعادل اختيار العينة التعديل لمتغير ما؟

## ✅ فحص سريع

<div class="quiz" data-quiz="statistics-101-adjustment-ar">
  <div class="quiz-q" data-answer="2">
    <p class="quiz-q__prompt">1. كيف تقرأ معاملًا في انحدار متعدد؟</p>
    <div class="quiz-q__options">
      <button class="quiz-q__opt" data-idx="0">الأثر الكلي لذلك المتغير وحده.</button>
      <button class="quiz-q__opt" data-idx="1">الارتباط بين ذلك المتغير والمخرَج.</button>
      <button class="quiz-q__opt" data-idx="2">التغير المرتبط بزيادة وحدة واحدة، مع تثبيت المنبئات الأخرى.</button>
      <button class="quiz-q__opt" data-idx="3">نسبة التباين التي يفسرها.</button>
    </div>
    <p class="quiz-q__feedback" hidden></p>
  </div>
  <div class="quiz-q" data-answer="1">
    <p class="quiz-q__prompt">2. لأي متغير يجب أن تعدّل؟</p>
    <div class="quiz-q__options">
      <button class="quiz-q__opt" data-idx="0">كل متغير في مجموعة البيانات.</button>
      <button class="quiz-q__opt" data-idx="1">سبب مقاس يؤثر في المنبئ والمخرَج معًا.</button>
      <button class="quiz-q__opt" data-idx="2">متغير قيس بعد العلاج وأنتجه العلاج.</button>
      <button class="quiz-q__opt" data-idx="3">المتغيرات الدالة إحصائيًا فقط.</button>
    </div>
    <p class="quiz-q__feedback" hidden></p>
  </div>
  <p class="quiz__summary" data-quiz-summary hidden></p>
</div>


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
